# Oil Region Selection with Bootstrap Risk Analysis

Selecting a drilling region under budget and downside-risk constraints.

**Result:** Region 2 was recommended: estimated mean profit RUB 536.4 million, 95% interval RUB 110 million–1.00 billion, and 0.3% loss risk.

**Methods:** regression, bootstrap, uncertainty estimation, risk analysis, business decision modelling.

> This portfolio version removes course-review correspondence and repetitive instructional text. The analysis, models, and reported metrics are based on the original completed project. The source datasets are not included in this repository.


## 1. Setup and data preparation

Three regional datasets are checked, deduplicated, and split into training and validation samples.


In [95]:
import pandas as pd

from IPython.display import display

from sklearn.linear_model import LinearRegression

from sklearn.model_selection import train_test_split

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import r2_score

from sklearn.preprocessing import StandardScaler

import numpy as np
from scipy import stats

from numpy.random import RandomState

import warnings
warnings.filterwarnings('ignore')


In [96]:
try:
    df_1 = pd.read_csv(
        '/Users/kolotukhin.md/Downloads/jupyter_notebook/7/geo_data_0.csv')
    df_2 = pd.read_csv(
        '/Users/kolotukhin.md/Downloads/jupyter_notebook/7/geo_data_1.csv')
    df_3 = pd.read_csv(
        '/Users/kolotukhin.md/Downloads/jupyter_notebook/7/geo_data_2.csv')
except:
    df_1 = pd.read_csv('https://code.s3.yandex.net/datasets/geo_data_0.csv')
    df_2 = pd.read_csv('https://code.s3.yandex.net/datasets/geo_data_1.csv')
    df_3 = pd.read_csv('https://code.s3.yandex.net/datasets/geo_data_2.csv')

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)


In [97]:
df_list = [df_1, df_2, df_3]
for i in df_list:
    display(i.head())
    print(i.info())


,id,f0,f1,f2,product
0,txEyH,0.705745,-0.497823,1.221170,105.280062
1,2acmU,1.334711,-0.340164,4.365080,73.037750
2,409Wp,1.022732,0.151990,1.419926,85.265647
3,iJLyR,-0.032172,0.139033,2.978566,168.620776
4,Xdl7t,1.988431,0.155413,4.751769,154.036647


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None


,id,f0,f1,f2,product
0,kBEdx,-15.001348,-8.276000,-0.005876,3.179103
1,62mP7,14.272088,-3.475083,0.999183,26.953261
2,vyE1P,6.263187,-5.948386,5.001160,134.766305
3,KcrkZ,-13.081196,-11.506057,4.999415,137.945408
4,AHL4O,12.702195,-8.147433,5.004363,134.766305


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None


,id,f0,f1,f2,product
0,fwXo0,-1.146987,0.963328,-0.828965,27.758673
1,WJtFt,0.262778,0.269839,-2.530187,56.069697
2,ovLUW,0.194587,0.289035,-5.586433,62.871910
3,q6cA6,2.236060,-0.553760,0.930038,114.572842
4,WPMUX,-0.515993,1.716266,5.899011,149.600746


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 5 columns):
 #   Column   Non-Null Count   Dtype  
---  ------   --------------   -----  
 0   id       100000 non-null  object 
 1   f0       100000 non-null  float64
 2   f1       100000 non-null  float64
 3   f2       100000 non-null  float64
 4   product  100000 non-null  float64
dtypes: float64(4), object(1)
memory usage: 3.8+ MB
None


In [98]:
# df_1.info()
# df_1.head()


In [99]:
# df_2.info()
# df_2.head()


In [100]:
# df_3.info()
# df_3.head()


In [101]:
for i in df_list:
    print(i.duplicated().value_counts())
    # print(i.duplicated().sum())


False    100000
dtype: int64
False    100000
dtype: int64
False    100000
dtype: int64


In [102]:
for i in df_list:

    print(i['id'].duplicated().value_counts())

    # print(i['id'].duplicated().sum())

    duplicates = i[i['id'].duplicated()].id.to_list()

    display(i[i['id'].isin(duplicates)].sort_values(by='id'))


False    99990
True        10
Name: id, dtype: int64


,id,f0,f1,f2,product
66136,74z30,1.084962,-0.312358,6.990771,127.643327
64022,74z30,0.741456,0.459229,5.153109,140.771492
51970,A5aEY,-0.180335,0.935548,-2.094773,33.020205
3389,A5aEY,-0.039949,0.156872,0.209861,89.249364
69163,AGS9W,-0.933795,0.116194,-3.655896,19.230453
42529,AGS9W,1.454747,-0.479651,0.683380,126.370504
931,HZww2,0.755284,0.368511,1.863211,30.681774
7530,HZww2,1.061194,-0.373969,10.430210,158.828695
63593,QcMuo,0.635635,-0.473422,0.862670,64.578675
1949,QcMuo,0.506563,-0.323775,-2.215583,75.496502


False    99996
True         4
Name: id, dtype: int64


,id,f0,f1,f2,product
5849,5ltQ6,-3.435401,-12.296043,1.999796,57.085625
84461,5ltQ6,18.213839,2.191999,3.993869,107.813044
1305,LHZR0,11.170835,-1.945066,3.002872,80.859783
41906,LHZR0,-8.989672,-4.286607,2.009139,57.085625
2721,bfPNe,-9.494442,-5.463692,4.006042,110.992147
82178,bfPNe,-6.202799,-4.820045,2.995107,84.038886
47591,wt4Uk,-9.091098,-8.109279,-0.002314,3.179103
82873,wt4Uk,10.259972,-9.376355,4.994297,134.766305


False    99996
True         4
Name: id, dtype: int64


,id,f0,f1,f2,product
45404,KUPhW,0.231846,-1.698941,4.990775,11.716299
55967,KUPhW,1.211150,3.176408,5.543540,132.831802
11449,VF7Jo,2.122656,-0.858275,5.746001,181.716817
49564,VF7Jo,-0.883115,0.560537,0.723601,136.233420
44378,Vcm5J,-1.229484,-2.439204,1.222909,137.968290
95090,Vcm5J,2.587702,1.986875,2.482245,92.327572
28039,xCHr8,1.633027,0.368135,-2.378367,6.120525
43233,xCHr8,-0.847066,2.101796,5.597130,184.388641


In [103]:

for i in df_list:
    i.drop_duplicates(subset=['id'], inplace=True)
    print(i['id'].duplicated().value_counts())
    i.drop(columns=['id'], axis=1, inplace=True)
    display(i.head())


False    99990
Name: id, dtype: int64


,f0,f1,f2,product
0,0.705745,-0.497823,1.221170,105.280062
1,1.334711,-0.340164,4.365080,73.037750
2,1.022732,0.151990,1.419926,85.265647
3,-0.032172,0.139033,2.978566,168.620776
4,1.988431,0.155413,4.751769,154.036647


False    99996
Name: id, dtype: int64


,f0,f1,f2,product
0,-15.001348,-8.276000,-0.005876,3.179103
1,14.272088,-3.475083,0.999183,26.953261
2,6.263187,-5.948386,5.001160,134.766305
3,-13.081196,-11.506057,4.999415,137.945408
4,12.702195,-8.147433,5.004363,134.766305


False    99996
Name: id, dtype: int64


,f0,f1,f2,product
0,-1.146987,0.963328,-0.828965,27.758673
1,0.262778,0.269839,-2.530187,56.069697
2,0.194587,0.289035,-5.586433,62.871910
3,2.236060,-0.553760,0.930038,114.572842
4,-0.515993,1.716266,5.899011,149.600746


In [104]:
def split_df(df):
    features = df.drop(['product'], axis=1)
    target = df['product']

    features_train, features_valid, target_train, target_valid = train_test_split(
        features, target, test_size=0.25, random_state=12345
    )

    for i in features_train, features_valid, target_train, target_valid:
        i.reset_index(drop=True, inplace=True)

    print(features_train.shape, features_valid.shape,
          target_train.shape, target_valid.shape)

    return features_train, features_valid, target_train, target_valid


features_train_1, features_valid_1, target_train_1, target_valid_1 = split_df(
    df_1)
features_train_2, features_valid_2, target_train_2, target_valid_2 = split_df(
    df_2)
features_train_3, features_valid_3, target_train_3, target_valid_3 = split_df(
    df_3)


(74992, 3) (24998, 3) (74992,) (24998,)
(74997, 3) (24999, 3) (74997,) (24999,)
(74997, 3) (24999, 3) (74997,) (24999,)


## 2. Production prediction

A separate linear regression model estimates well output for each region.


In [105]:
pd.options.mode.chained_assignment = None

numeric = ['f0', 'f1', 'f2']

scaler = StandardScaler()


In [106]:
scaler.fit(features_train_1[numeric])

features_train_1[numeric] = scaler.transform(features_train_1[numeric])
features_valid_1[numeric] = scaler.transform(features_valid_1[numeric])


In [107]:
scaler.fit(features_train_2[numeric])

features_train_2[numeric] = scaler.transform(features_train_2[numeric])
features_valid_2[numeric] = scaler.transform(features_valid_2[numeric])


In [108]:
scaler.fit(features_train_3[numeric])

features_train_3[numeric] = scaler.transform(features_train_3[numeric])
features_valid_3[numeric] = scaler.transform(features_valid_3[numeric])


In [109]:
def calculate_metrics(features_train, features_valid, target_train, target_valid, predicted_valid, model):
    model_metrics = []
    R2_score = r2_score(target_valid, predicted_valid)
    MSE = mean_squared_error(target_valid, predicted_valid)
    RMSE = MSE**0.5
    MAE = mean_absolute_error(target_valid, predicted_valid)
    model_metrics.append([R2_score, MSE, RMSE, MAE])

    return np.hstack((model_metrics))



def metrics_list():
    metrics = ['R2_score', 'MSE', 'RMSE', 'MAE']

    return metrics


In [110]:
def model_LinearRegression(features_train, features_valid, target_train, target_valid):
    model = LinearRegression()
    model.fit(features_train, target_train)

    predicted_valid = model.predict(features_valid)

    model_metrics = calculate_metrics(features_train, features_valid,
                                      target_train, target_valid,
                                      predicted_valid, model
                                      )

    return predicted_valid, model_metrics


model_metrics = []

predicted_valid_1, metrics = model_LinearRegression(features_train_1, features_valid_1,
                                                    target_train_1, target_valid_1
                                                    )
model_metrics.append(np.hstack(([metrics, predicted_valid_1.mean()])))

predicted_valid_2, metrics = model_LinearRegression(features_train_2, features_valid_2,
                                                    target_train_2, target_valid_2
                                                    )
model_metrics.append(np.hstack(([metrics, predicted_valid_2.mean()])))

predicted_valid_3, metrics = model_LinearRegression(features_train_3, features_valid_3,
                                                    target_train_3, target_valid_3
                                                    )
model_metrics.append(np.hstack(([metrics, predicted_valid_3.mean()])))

model_metrics = pd.DataFrame(model_metrics, columns=['R2_score', 'MSE', 'RMSE', 'MAE', 'Predicted_mean'],
                             index=['Region_1', 'Region_2', 'Region_3'])
display(model_metrics.T)


,Region_1,Region_2,Region_3
R2_score,0.272392,0.999622,0.195562
MSE,1432.889531,0.795770,1606.073812
RMSE,37.853527,0.892059,40.075851
MAE,31.141029,0.719353,32.831390
Predicted_mean,92.789156,69.178320,94.865725


In [111]:
model_1 = LinearRegression()

model_1.fit(features_train_1, target_train_1)

predictions_valid_1 = model_1.predict(features_valid_1)

result_1 = mean_squared_error(target_valid_1, predictions_valid_1) ** 0.5

print('Region 1:')
print('Linear Regression validation RMSE:', round(result_1, 2))
print('Mean predicted reserve (thousand barrels):',
      round(predictions_valid_1.mean(), 2))
print('Mean observed reserve (thousand barrels): ',
      round(df_1['product'].mean(), 2))


In [112]:
model_2 = LinearRegression()

model_2.fit(features_train_2, target_train_2)

predictions_valid_2 = model_2.predict(features_valid_2)

result_2 = mean_squared_error(target_valid_2, predictions_valid_2) ** 0.5

print('Region 2:')
print("Linear Regression validation RMSE:", round(result_2, 2))
print('Mean predicted reserve (thousand barrels):',
      round(predictions_valid_2.mean(), 2))
print('Mean observed reserve (thousand barrels): ',
      round(df_2['product'].mean(), 2))


In [113]:
model_3 = LinearRegression()

model_3.fit(features_train_3, target_train_3)

predictions_valid_3 = model_3.predict(features_valid_3)

result_3 = mean_squared_error(target_valid_3, predictions_valid_3) ** 0.5

print('Region 3:')
print("Linear Regression validation RMSE:", round(result_3, 2))
print('Mean predicted reserve (thousand barrels):',
      round(predictions_valid_3.mean(), 2))
print('Mean observed reserve (thousand barrels): ',
      round(df_3['product'].mean(), 2))


## 3. Economic assumptions

The break-even reserve threshold is derived from the exploration budget, selected-well count, and unit revenue.


In [114]:
BUDGET = pow(10, 10)

CHOOSEN_POINTS = 500

BEST_POINTS = 200

BARREL_INCOME = 450

UNIT_BARREL_INCOME = BARREL_INCOME * pow(10, 3)


In [115]:
minimum_volume_barrel = (BUDGET / BEST_POINTS / UNIT_BARREL_INCOME)

print('Break-even reserve per well (thousand barrels): {:.2f}'
      .format(minimum_volume_barrel)
      )
print()
print('Region 1:')
print('Linear Regression validation RMSE:', round(result_1, 2))
print('Mean predicted reserve (thousand barrels):',
      round(predictions_valid_1.mean(), 2))
print('Mean observed reserve (thousand barrels): ',
      round(df_1['product'].mean(), 2))
print()
print('Region 2:')
print("Linear Regression validation RMSE:", round(result_2, 2))
print('Mean predicted reserve (thousand barrels):',
      round(predictions_valid_2.mean(), 2))
print('Mean observed reserve (thousand barrels): ',
      round(df_2['product'].mean(), 2))
print()
print('Region 3:')
print("Linear Regression validation RMSE:", round(result_3, 2))
print('Mean predicted reserve (thousand barrels):',
      round(predictions_valid_3.mean(), 2))
print('Mean observed reserve (thousand barrels): ',
      round(df_3['product'].mean(), 2))


## 4. Profit function

Predicted wells are ranked and the realised reserves of the selected wells are used to estimate profit.


In [116]:
pred_valid_1 = pd.Series(predictions_valid_1, index=target_valid_1.index)
pred_valid_2 = pd.Series(predictions_valid_2, index=target_valid_2.index)
pred_valid_3 = pd.Series(predictions_valid_3, index=target_valid_3.index)


In [117]:
max_200_1 = pred_valid_1[:BEST_POINTS]
max_200_2 = pred_valid_2[:BEST_POINTS]
max_200_3 = pred_valid_3[:BEST_POINTS]

print('Potential total reserve:')
print('for the top 200 wells in Region 1: {:.2f} million barrels'.
      format(max_200_1.sum() / pow(10, 3))
      )
print('for the top 200 wells in Region 2: {:.2f} million barrels'.
      format(max_200_2.sum() / pow(10, 3))
      )
print('for the top 200 wells in Region 3: {:.2f} million barrels'.
      format(max_200_3.sum() / pow(10, 3))
      )


In [118]:
def income(target, valid, count):
    valid_sorted = valid.sort_values(ascending=False)
    selected = target.loc[valid_sorted.index][:count]
    profit = UNIT_BARREL_INCOME * selected.sum() - BUDGET
    return profit


print('Profit from the top 200 wells in Region 1:',
      round(income(target_valid_1, pred_valid_1, BEST_POINTS) / pow(10, 9), 2), 'RUB billion')
print('Profit from the top 200 wells in Region 2:',
      round(income(target_valid_2, pred_valid_2, BEST_POINTS) / pow(10, 9), 2), 'RUB billion')
print('Profit from the top 200 wells in Region 3:',
      round(income(target_valid_3, pred_valid_3, BEST_POINTS) / pow(10, 9), 2), 'RUB billion')


## 5. Bootstrap risk analysis

Bootstrap resampling estimates expected profit, the 95% uncertainty interval, and the probability of loss for each region.


In [119]:
state = np.random.RandomState(12345)

def bootstrap_profit(target, predictions, n_samples=1000):
    profits = []
    for _ in range(n_samples):
        target_sample = target.sample(n=CHOOSEN_POINTS, replace=True, random_state=state)
        prediction_sample = predictions.loc[target_sample.index]
        profits.append(income(target_sample, prediction_sample, BEST_POINTS))
    profits = pd.Series(profits)
    return {
        "mean_profit": profits.mean(),
        "lower_95": profits.quantile(0.025),
        "upper_95": profits.quantile(0.975),
        "loss_risk_pct": (profits < 0).mean() * 100,
    }

bootstrap_results = {
    "Region 1": bootstrap_profit(target_valid_1, pred_valid_1),
    "Region 2": bootstrap_profit(target_valid_2, pred_valid_2),
    "Region 3": bootstrap_profit(target_valid_3, pred_valid_3),
}
pd.DataFrame(bootstrap_results).T


## Conclusion

Region 2 is the only candidate comfortably below the 2.5% loss-risk ceiling and also has the highest estimated mean profit. The recommendation depends on the stated synthetic-data assumptions and should be revisited if drilling cost or unit revenue changes.
